# Lab 01 — 同一份數據、兩種視角

**課程**：player-behavior-analytics（玩家行為分析）／第 1 課 從莊家優勢到玩家中心建模（第 4 節 Colab 練習）
**目的**：用「新分析方法」親眼看一次範式轉移——同一份模擬數據，聚合視角與個體視角產出完全不同的洞察。
**方式**：全程以 Gemini（AI 助手）產生程式碼——把每個任務的自然語言描述貼給 Gemini，再把生成的程式碼貼進下方 code cell 執行；**重點是觀察結果**，不是寫程式。每個任務下方附參考程式碼，可先自行生成再比對。

> 執行：在 Google Colab 上傳本 .ipynb（或直接開啟），Runtime → Run all。本範本內建模擬數據產生器，無需上傳檔案。

## 0. 載入數據

執行下方 cell 產生模擬數據（10 位玩家 × 3 天 × 每日 2 session × 每 session 30 局 = 1,800 筆）。欄位：

| 欄位 | 說明 |
|------|------|
| player_id | 玩家編號（P00–P09） |
| session_id | 會話編號 |
| table_id | 桌號（T1–T2） |
| timestamp | 下注時間 |
| round_no | 局號（每 session 1–30） |
| bet_type | 下注類型：Banker / Player / Tie |
| bet_amount | 下注金額 |
| outcome | 開牌結果 |
| payout | 派彩（輸局為 0） |
| is_win | 是否贏局 |

> 教學簡化：本 Lab 隱藏每位玩家的「真實原型」欄位（第 4 課才揭露），讓你純粹從數據觀察。

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt

ARCHETYPES = ["consistent", "chaser", "momentum", "fallacy", "volatile"]


def gen_baccarat(seed=42, n_players=10, days=3, sessions=2, rounds=30):
    '''模擬智慧娛樂桌百家樂下注紀錄（教學用生成器）。

    原型依序循環：consistent 紀律型 / chaser 追注型 / momentum 順勢型 /
    fallacy 謬誤型 / volatile 波動型。
    簡化：每位玩家的每局結果獨立抽籤（真實同桌玩家共享開牌結果）。
    '''
    rng = np.random.default_rng(seed)
    BT = ["Banker", "Player", "Tie"]
    P_OUT = [0.4586, 0.4462, 0.0952]   # 8 副牌百家樂開牌機率
    P_BET = [0.50, 0.45, 0.05]         # 下注類型偏好
    BASE = {"consistent": 500, "chaser": 500, "momentum": 500,
            "fallacy": 500, "volatile": 300}
    PATS = {"consistent": [(14, 0), (20, 0)],   # 各原型偏好時段
            "chaser": [(20, 0), (22, 0)],
            "momentum": [(14, 0), (16, 0)],
            "fallacy": [(18, 0), (20, 0)],
            "volatile": None}                   # None = 隨機時段
    first = dt.date(2026, 6, 1)

    def pay(bt, oc, amt):
        if bt == "Banker" and oc == "Banker":
            return round(amt * 0.95, 2)   # 莊贏抽 5% 佣金
        if bt == "Player" and oc == "Player":
            return amt
        if bt == "Tie" and oc == "Tie":
            return round(amt * 8.0, 2)    # 和局賠 8 倍
        return 0.0

    rows = []
    for p in range(n_players):
        arch = ARCHETYPES[p % len(ARCHETYPES)]
        cur, prev_win = BASE[arch], None
        streak_out, streak_len, last_bt = None, 0, None
        for d in range(days):
            day = first + dt.timedelta(days=d)
            for s in range(sessions):
                pat = PATS[arch]
                if pat:
                    hh, mm = pat[s % len(pat)]
                else:
                    hh, mm = int(rng.integers(12, 24)), int(rng.integers(0, 60))
                t0 = dt.datetime(day.year, day.month, day.day, hh, mm, 0)
                for r in range(rounds):
                    oc = rng.choice(BT, p=P_OUT)
                    # --- 下注類型 ---
                    if arch == "fallacy" and streak_len >= 3 and oc == streak_out:
                        # 相信「平衡定律」：連開 3 次同路後轉押另一邊
                        bt = (rng.choice(BT[:2]) if streak_out == "Tie"
                              else ("Player" if streak_out == "Banker" else "Banker"))
                    elif arch == "fallacy":
                        bt = last_bt if last_bt else rng.choice(BT, p=P_BET)
                    else:
                        bt = rng.choice(BT, p=P_BET)
                    # --- 下注金額 ---
                    if arch == "consistent" or arch == "fallacy":
                        cur = BASE[arch]
                    elif arch == "chaser":    # 輸後加注追趕，贏後回到 base
                        cur = BASE[arch] if prev_win is not False else min(BASE[arch] * 3, int(cur * 1.5))
                    elif arch == "momentum":  # 贏後順勢加注，輸後回到 base
                        cur = min(BASE[arch] * 3, int(cur * 1.4)) if prev_win else BASE[arch]
                    else:                     # volatile：忽大忽小
                        cur = int(rng.integers(1, 11)) * 100
                    rows.append({
                        "player_id": "P%02d" % p, "archetype": arch,
                        "session_id": "P%02d-D%d-S%d" % (p, d + 1, s + 1),
                        "table_id": "T%d" % (p % 2 + 1),
                        "timestamp": t0 + dt.timedelta(seconds=45 * r),
                        "round_no": r + 1, "bet_type": bt, "bet_amount": cur,
                        "outcome": oc, "payout": pay(bt, oc, cur)})
                    prev_win = pay(bt, oc, cur) > 0
                    streak_out, streak_len = (oc, streak_len + 1) if oc == streak_out else (oc, 1)
                    last_bt = bt
    df = pd.DataFrame(rows)
    df["is_win"] = df["payout"] > 0
    return df
import pandas as pd
import numpy as np
import datetime as dt

ARCHETYPES = ["consistent", "chaser", "momentum", "fallacy", "volatile"]


def gen_baccarat(seed=42, n_players=10, days=3, sessions=2, rounds=30):
    """模擬智慧娛樂桌百家樂下注紀錄（教學用生成器）。

    原型依序循環：consistent 紀律型 / chaser 追注型 / momentum 順勢型 /
    fallacy 謬誤型 / volatile 波動型。
    簡化：每位玩家的每局結果獨立抽籤（真實同桌玩家共享開牌結果）。
    """
    rng = np.random.default_rng(seed)
    BT = ["Banker", "Player", "Tie"]
    P_OUT = [0.4586, 0.4462, 0.0952]   # 8 副牌百家樂開牌機率
    P_BET = [0.50, 0.45, 0.05]         # 下注類型偏好
    BASE = {"consistent": 500, "chaser": 500, "momentum": 500,
            "fallacy": 500, "volatile": 300}
    PATS = {"consistent": [(14, 0), (20, 0)],   # 各原型偏好時段
            "chaser": [(20, 0), (22, 0)],
            "momentum": [(14, 0), (16, 0)],
            "fallacy": [(18, 0), (20, 0)],
            "volatile": None}                   # None = 隨機時段
    first = dt.date(2026, 6, 1)

    def pay(bt, oc, amt):
        if bt == "Banker" and oc == "Banker":
            return round(amt * 0.95, 2)   # 莊贏抽 5% 佣金
        if bt == "Player" and oc == "Player":
            return amt
        if bt == "Tie" and oc == "Tie":
            return round(amt * 8.0, 2)    # 和局賠 8 倍
        return 0.0

    rows = []
    for p in range(n_players):
        arch = ARCHETYPES[p % len(ARCHETYPES)]
        cur, prev_win = BASE[arch], None
        streak_out, streak_len, last_bt = None, 0, None
        for d in range(days):
            day = first + dt.timedelta(days=d)
            for s in range(sessions):
                pat = PATS[arch]
                if pat:
                    hh, mm = pat[s % len(pat)]
                else:
                    hh, mm = int(rng.integers(12, 24)), int(rng.integers(0, 60))
                t0 = dt.datetime(day.year, day.month, day.day, hh, mm, 0)
                for r in range(rounds):
                    oc = rng.choice(BT, p=P_OUT)
                    # --- 下注類型 ---
                    if arch == "fallacy" and streak_len >= 3 and oc == streak_out:
                        # 相信「平衡定律」：連開 3 次同路後轉押另一邊
                        bt = (rng.choice(BT[:2]) if streak_out == "Tie"
                              else ("Player" if streak_out == "Banker" else "Banker"))
                    elif arch == "fallacy":
                        bt = last_bt if last_bt else rng.choice(BT, p=P_BET)
                    else:
                        bt = rng.choice(BT, p=P_BET)
                    # --- 下注金額 ---
                    if arch == "consistent" or arch == "fallacy":
                        cur = BASE[arch]
                    elif arch == "chaser":    # 輸後加注追趕，贏後回到 base
                        cur = BASE[arch] if prev_win is not False else min(BASE[arch] * 3, int(cur * 1.5))
                    elif arch == "momentum":  # 贏後順勢加注，輸後回到 base
                        cur = min(BASE[arch] * 3, int(cur * 1.4)) if prev_win else BASE[arch]
                    else:                     # volatile：忽大忽小
                        cur = int(rng.integers(1, 11)) * 100
                    rows.append({
                        "player_id": "P%02d" % p, "archetype": arch,
                        "session_id": "P%02d-D%d-S%d" % (p, d + 1, s + 1),
                        "table_id": "T%d" % (p % 2 + 1),
                        "timestamp": t0 + dt.timedelta(seconds=45 * r),
                        "round_no": r + 1, "bet_type": bt, "bet_amount": cur,
                        "outcome": oc, "payout": pay(bt, oc, cur)})
                    prev_win = pay(bt, oc, cur) > 0
                    streak_out, streak_len = (oc, streak_len + 1) if oc == streak_out else (oc, 1)
                    last_bt = bt
    df = pd.DataFrame(rows)
    df["is_win"] = df["payout"] > 0
    return df

df = gen_baccarat(seed=42)
df = df.drop(columns=["archetype"])   # 教學簡化：先不看真實原型
df.head()


## Task A — 聚合視角（舊方法）

把以下描述貼給 Gemini（並附上「數據已在變數 df」）：

> 「這個 DataFrame 是智慧娛樂桌的百家樂下注紀錄（欄位：player_id / session_id / table_id / timestamp / round_no / bet_type / bet_amount / outcome / payout / is_win）。請計算每桌每日的 GGR（下注總額減派彩總額），並輸出摘要。」

下方為參考程式碼，可在 Gemini 生成後對照。

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt

ARCHETYPES = ["consistent", "chaser", "momentum", "fallacy", "volatile"]


def gen_baccarat(seed=42, n_players=10, days=3, sessions=2, rounds=30):
    '''模擬智慧娛樂桌百家樂下注紀錄（教學用生成器）。

    原型依序循環：consistent 紀律型 / chaser 追注型 / momentum 順勢型 /
    fallacy 謬誤型 / volatile 波動型。
    簡化：每位玩家的每局結果獨立抽籤（真實同桌玩家共享開牌結果）。
    '''
    rng = np.random.default_rng(seed)
    BT = ["Banker", "Player", "Tie"]
    P_OUT = [0.4586, 0.4462, 0.0952]   # 8 副牌百家樂開牌機率
    P_BET = [0.50, 0.45, 0.05]         # 下注類型偏好
    BASE = {"consistent": 500, "chaser": 500, "momentum": 500,
            "fallacy": 500, "volatile": 300}
    PATS = {"consistent": [(14, 0), (20, 0)],   # 各原型偏好時段
            "chaser": [(20, 0), (22, 0)],
            "momentum": [(14, 0), (16, 0)],
            "fallacy": [(18, 0), (20, 0)],
            "volatile": None}                   # None = 隨機時段
    first = dt.date(2026, 6, 1)

    def pay(bt, oc, amt):
        if bt == "Banker" and oc == "Banker":
            return round(amt * 0.95, 2)   # 莊贏抽 5% 佣金
        if bt == "Player" and oc == "Player":
            return amt
        if bt == "Tie" and oc == "Tie":
            return round(amt * 8.0, 2)    # 和局賠 8 倍
        return 0.0

    rows = []
    for p in range(n_players):
        arch = ARCHETYPES[p % len(ARCHETYPES)]
        cur, prev_win = BASE[arch], None
        streak_out, streak_len, last_bt = None, 0, None
        for d in range(days):
            day = first + dt.timedelta(days=d)
            for s in range(sessions):
                pat = PATS[arch]
                if pat:
                    hh, mm = pat[s % len(pat)]
                else:
                    hh, mm = int(rng.integers(12, 24)), int(rng.integers(0, 60))
                t0 = dt.datetime(day.year, day.month, day.day, hh, mm, 0)
                for r in range(rounds):
                    oc = rng.choice(BT, p=P_OUT)
                    # --- 下注類型 ---
                    if arch == "fallacy" and streak_len >= 3 and oc == streak_out:
                        # 相信「平衡定律」：連開 3 次同路後轉押另一邊
                        bt = (rng.choice(BT[:2]) if streak_out == "Tie"
                              else ("Player" if streak_out == "Banker" else "Banker"))
                    elif arch == "fallacy":
                        bt = last_bt if last_bt else rng.choice(BT, p=P_BET)
                    else:
                        bt = rng.choice(BT, p=P_BET)
                    # --- 下注金額 ---
                    if arch == "consistent" or arch == "fallacy":
                        cur = BASE[arch]
                    elif arch == "chaser":    # 輸後加注追趕，贏後回到 base
                        cur = BASE[arch] if prev_win is not False else min(BASE[arch] * 3, int(cur * 1.5))
                    elif arch == "momentum":  # 贏後順勢加注，輸後回到 base
                        cur = min(BASE[arch] * 3, int(cur * 1.4)) if prev_win else BASE[arch]
                    else:                     # volatile：忽大忽小
                        cur = int(rng.integers(1, 11)) * 100
                    rows.append({
                        "player_id": "P%02d" % p, "archetype": arch,
                        "session_id": "P%02d-D%d-S%d" % (p, d + 1, s + 1),
                        "table_id": "T%d" % (p % 2 + 1),
                        "timestamp": t0 + dt.timedelta(seconds=45 * r),
                        "round_no": r + 1, "bet_type": bt, "bet_amount": cur,
                        "outcome": oc, "payout": pay(bt, oc, cur)})
                    prev_win = pay(bt, oc, cur) > 0
                    streak_out, streak_len = (oc, streak_len + 1) if oc == streak_out else (oc, 1)
                    last_bt = bt
    df = pd.DataFrame(rows)
    df["is_win"] = df["payout"] > 0
    return df
# Task A 參考程式碼：聚合視角——每桌每日 GGR
df["date"] = pd.to_datetime(df["timestamp"]).dt.date
ggr = (df.groupby(["table_id", "date"])
         .agg(bet_total=("bet_amount", "sum"),
              payout_total=("payout", "sum"))
         .assign(ggr=lambda x: x.bet_total - x.payout_total))
ggr

**觀察 Task A**：
- 輸出只有每桌每日 6 個數字——在舊視角下，10 位玩家「完全一樣」
- 這正是莊家優勢時代營運者能看到的一切：只知道總量，不知道「誰」在玩、「如何」在玩

## Task B — 個體視角（新方法）

在 Gemini 輸入：

> 「請按玩家分組，輸出每位玩家的：session 數、總下注、平均下注、下注金額標準差、最常下注類型；並畫出其中 3 位玩家的逐局下注折線圖。」

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt

ARCHETYPES = ["consistent", "chaser", "momentum", "fallacy", "volatile"]


def gen_baccarat(seed=42, n_players=10, days=3, sessions=2, rounds=30):
    '''模擬智慧娛樂桌百家樂下注紀錄（教學用生成器）。

    原型依序循環：consistent 紀律型 / chaser 追注型 / momentum 順勢型 /
    fallacy 謬誤型 / volatile 波動型。
    簡化：每位玩家的每局結果獨立抽籤（真實同桌玩家共享開牌結果）。
    '''
    rng = np.random.default_rng(seed)
    BT = ["Banker", "Player", "Tie"]
    P_OUT = [0.4586, 0.4462, 0.0952]   # 8 副牌百家樂開牌機率
    P_BET = [0.50, 0.45, 0.05]         # 下注類型偏好
    BASE = {"consistent": 500, "chaser": 500, "momentum": 500,
            "fallacy": 500, "volatile": 300}
    PATS = {"consistent": [(14, 0), (20, 0)],   # 各原型偏好時段
            "chaser": [(20, 0), (22, 0)],
            "momentum": [(14, 0), (16, 0)],
            "fallacy": [(18, 0), (20, 0)],
            "volatile": None}                   # None = 隨機時段
    first = dt.date(2026, 6, 1)

    def pay(bt, oc, amt):
        if bt == "Banker" and oc == "Banker":
            return round(amt * 0.95, 2)   # 莊贏抽 5% 佣金
        if bt == "Player" and oc == "Player":
            return amt
        if bt == "Tie" and oc == "Tie":
            return round(amt * 8.0, 2)    # 和局賠 8 倍
        return 0.0

    rows = []
    for p in range(n_players):
        arch = ARCHETYPES[p % len(ARCHETYPES)]
        cur, prev_win = BASE[arch], None
        streak_out, streak_len, last_bt = None, 0, None
        for d in range(days):
            day = first + dt.timedelta(days=d)
            for s in range(sessions):
                pat = PATS[arch]
                if pat:
                    hh, mm = pat[s % len(pat)]
                else:
                    hh, mm = int(rng.integers(12, 24)), int(rng.integers(0, 60))
                t0 = dt.datetime(day.year, day.month, day.day, hh, mm, 0)
                for r in range(rounds):
                    oc = rng.choice(BT, p=P_OUT)
                    # --- 下注類型 ---
                    if arch == "fallacy" and streak_len >= 3 and oc == streak_out:
                        # 相信「平衡定律」：連開 3 次同路後轉押另一邊
                        bt = (rng.choice(BT[:2]) if streak_out == "Tie"
                              else ("Player" if streak_out == "Banker" else "Banker"))
                    elif arch == "fallacy":
                        bt = last_bt if last_bt else rng.choice(BT, p=P_BET)
                    else:
                        bt = rng.choice(BT, p=P_BET)
                    # --- 下注金額 ---
                    if arch == "consistent" or arch == "fallacy":
                        cur = BASE[arch]
                    elif arch == "chaser":    # 輸後加注追趕，贏後回到 base
                        cur = BASE[arch] if prev_win is not False else min(BASE[arch] * 3, int(cur * 1.5))
                    elif arch == "momentum":  # 贏後順勢加注，輸後回到 base
                        cur = min(BASE[arch] * 3, int(cur * 1.4)) if prev_win else BASE[arch]
                    else:                     # volatile：忽大忽小
                        cur = int(rng.integers(1, 11)) * 100
                    rows.append({
                        "player_id": "P%02d" % p, "archetype": arch,
                        "session_id": "P%02d-D%d-S%d" % (p, d + 1, s + 1),
                        "table_id": "T%d" % (p % 2 + 1),
                        "timestamp": t0 + dt.timedelta(seconds=45 * r),
                        "round_no": r + 1, "bet_type": bt, "bet_amount": cur,
                        "outcome": oc, "payout": pay(bt, oc, cur)})
                    prev_win = pay(bt, oc, cur) > 0
                    streak_out, streak_len = (oc, streak_len + 1) if oc == streak_out else (oc, 1)
                    last_bt = bt
    df = pd.DataFrame(rows)
    df["is_win"] = df["payout"] > 0
    return df
# Task B 參考程式碼：個體視角——每位玩家的行為輪廓
stats = (df.groupby("player_id")
           .agg(sessions=("session_id", "nunique"),
                total_bet=("bet_amount", "sum"),
                avg_bet=("bet_amount", "mean"),
                std_bet=("bet_amount", "std"),
                top_bet_type=("bet_type", lambda s: s.mode()[0]))
           .assign(cv_bet=lambda x: x.std_bet / x.avg_bet))
stats.round(0)

# 其中 3 位玩家的逐局下注折線圖
import matplotlib.pyplot as plt

for pid in ["P00", "P01", "P04"]:
    sub = df[df["player_id"] == pid].sort_values("timestamp").reset_index(drop=True)
    plt.plot(sub["bet_amount"], label=pid)
plt.legend()
plt.title("Bet amount per round (3 players)")
plt.xlabel("round"); plt.ylabel("bet amount")
plt.show()

**觀察 Task B**：
- 玩家輪廓瞬間分離：P00 固定金額型（std = 0）、P01 輸後加注型（折線呈台階式爬升）、P04 忽大忽小型（鋸齒狀）
- 折線圖顯示「順序結構」——總下注接近的玩家（本例 P05 約 9 萬、P04 約 10 萬），行為節奏完全不同

**對照與討論**：
- 同一份數據：Task A 一組總數、Task B 十個輪廓——差距就是智慧娛樂桌＋玩家中心建模的價值
- 三個轉變：被動報告（A）→ 主動預測（B 的序列可推測下一步）；大眾行銷（A）→ 個人化（B 的分群）
- 延伸思考：這種數據還可應用在哪裡？（詐賭偵測、洗碼分析、問題博弈預警）；玩家中心建模有哪些風險？（隱私、公平性、問題博弈——第 5 課深入）